# CSV Fix Prompt

This notebook provides a simple prompt-driven flow to fix rows with problems and write changes back to the input CSV.

Assumptions:
- you already have `report_data` from `/jobs/{job_id}/result` saved as JSON;
- the CSV is the same one used in the job;
- row indexes in the report are 0-based (header is not counted).


In [ ]:
from pathlib import Path
import json

import pandas as pd
from IPython.display import Markdown, display

CSV_PATH = Path("./gc-test.csv")
REPORT_JSON_PATH = Path("../notebook_downloads/REPLACE_WITH_JOB_result.json")

if not CSV_PATH.exists():
    raise FileNotFoundError(f"Missing CSV: {CSV_PATH}")
if not REPORT_JSON_PATH.exists():
    raise FileNotFoundError(f"Missing report JSON: {REPORT_JSON_PATH}")

report_data = json.loads(REPORT_JSON_PATH.read_text(encoding="utf-8"))
df = pd.read_csv(CSV_PATH, dtype=str).fillna("")

display(Markdown(f"**CSV:** `{CSV_PATH.resolve()}`"))
display(Markdown(f"**Report JSON:** `{REPORT_JSON_PATH.resolve()}`"))
display(Markdown("## Preview of issues"))

def _line_number(row_index: int | None) -> str:
    if row_index is None:
        return "-"
    return str(row_index + 2)


issue_rows = []
for code, occurrences in report_data.get("grouped_problems", {}).items():
    for occurrence in occurrences:
        issue_rows.append(
            {
                "code": code,
                "row_index": occurrence.get("row_index"),
                "line_number": _line_number(occurrence.get("row_index")),
                "item": occurrence.get("item"),
                "descricao": occurrence.get("descricao"),
                "field": occurrence.get("field"),
                "message": occurrence.get("message"),
            }
        )

issues_df = pd.DataFrame(issue_rows)
display(issues_df if not issues_df.empty else pd.DataFrame([{"status": "No issues found"}]))


In [ ]:
def apply_fix(row_index: int, column: str, new_value: str) -> None:
    if row_index < 0 or row_index >= len(df):
        raise ValueError(f"Row index out of range: {row_index}")
    if column not in df.columns:
        raise ValueError(f"Unknown column: {column}")
    df.at[row_index, column] = new_value


def prompt_loop() -> None:
    print("Type 'q' to quit. Use row_index from the issues table above.")
    while True:
        row_input = input("Row index to fix (0-based): ").strip()
        if row_input.lower() == "q":
            break
        if not row_input.isdigit():
            print("Invalid row index. Try again.")
            continue

        row_index = int(row_input)
        column = input("Column to update (exact header): ").strip()
        new_value = input("New value: ").strip()

        try:
            apply_fix(row_index, column, new_value)
        except ValueError as exc:
            print(f"Error: {exc}")
            continue

        print("Updated row preview:")
        display(df.iloc[[row_index]])

    backup_path = CSV_PATH.with_suffix(".bak.csv")
    original_df = pd.read_csv(CSV_PATH, dtype=str).fillna("")
    original_df.to_csv(backup_path, index=False)
    df.to_csv(CSV_PATH, index=False)
    print(f"Saved CSV to {CSV_PATH}")
    print(f"Backup CSV to {backup_path}")


prompt_loop()
